<a href="https://colab.research.google.com/github/lightcode01-oss/IYB_INTERNSHIP/blob/main/Day_24_Model_Explainability_Feature_Importance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day 24: Model Explainability and Feature Importance
## YBI Foundation — Python AI & Data Science Internship
### Project: AI-Based Student Performance Prediction System

---

### 1. Day 24 Objectives & Overviews

In Days 14–23, we evaluated candidate algorithms, tuned hyperparameters, selected a champion Decision Tree model, built an automated validation suite, deployed an interactive Streamlit application (`app.py`), and established a monitoring and evaluation dashboard.

Today's milestone (**Day 24**) focuses on **Model Explainability and Feature Importance**:
- **Global Explainability (Permutation Importance):** Quantify which student features most strongly influence generalization performance (Macro F1) on the authentic held-out test dataset ($N=130$).
- **Native Model Importance:** Extract and aggregate the champion Decision Tree's Mean Decrease in Impurity (Gini importance) across preprocessed feature spaces.
- **Methodological Comparison:** Contrast test-set permutation importance with training-set native tree importance to distinguish genuine predictive utility from training-set split bias.
- **Local Interpretability:** Trace exact decision tree split paths and compare individual student attributes against cohort baselines for transparent early-warning feedback.
- **Target Leakage & Ethical Safeguards:** Rigorously enforce the exclusion of final grade `G3` and derived label `performance_category` from all feature inputs, and articulate why importance scores do not imply causation.

> **Core Constraint:** We do NOT retrain or replace the champion model (`artifacts/student_performance_pipeline.joblib`). All explanations evaluate the authentic pre-trained pipeline produced during Day 17.

---
### 2. Environment Setup and Project Path Resolution

We configure the environment to support both local workspace execution and Google Colab environments.

In [ ]:
import os
import sys
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Resolve Project Root (works locally and in Colab)
PROJECT_ROOT = Path(".").resolve()
if (PROJECT_ROOT / "artifacts").exists():
    pass
elif (PROJECT_ROOT.parent / "artifacts").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

pipeline_path = PROJECT_ROOT / "artifacts" / "student_performance_pipeline.joblib"
metadata_path = PROJECT_ROOT / "artifacts" / "student_performance_metadata.json"
data_path = PROJECT_ROOT / "day_16_error_analysis.csv"

print("=" * 70)
print("DAY 24: ENVIRONMENT & ARTIFACT DISCOVERY")
print("=" * 70)
print(f"Project Root Resolved: OK")
print(f"Pipeline Artifact Found: {pipeline_path.exists()} ({pipeline_path.name})")
print(f"Metadata Artifact Found: {metadata_path.exists()} ({metadata_path.name})")
print(f"Evaluation Dataset Found: {data_path.exists()} ({data_path.name})")

from src.explainability import (
    load_explainability_pipeline,
    load_reference_dataset,
    validate_feature_schema,
    compute_permutation_importance,
    compute_native_feature_importance,
    explain_individual_prediction,
    plot_permutation_importance,
    plot_native_importance,
    CANONICAL_FEATURES,
    FORBIDDEN_LEAKAGE_COLUMNS
)
print("Explainability Module: OK (src.explainability)")

DAY 24: ENVIRONMENT & ARTIFACT DISCOVERY
Project Root Resolved: OK
Pipeline Artifact Found: False (student_performance_pipeline.joblib)
Metadata Artifact Found: False (student_performance_metadata.json)
Evaluation Dataset Found: False (day_16_error_analysis.csv)


ModuleNotFoundError: No module named 'src'

---
### 3. Loading Model Pipeline and Verifying Preprocessing Architecture

We inspect the saved model artifact to confirm whether it contains an isolated estimator or an end-to-end scikit-learn `Pipeline` encapsulating preprocessing.

In [ ]:
pipeline = load_explainability_pipeline(pipeline_path)
with open(metadata_path, "r", encoding="utf-8") as f:
    metadata = json.load(f)

print(f"Loaded Object Type: {type(pipeline)}")
print(f"Pipeline Steps: {list(pipeline.named_steps.keys())}")

preprocessor = pipeline.named_steps["preprocessor"]
classifier = pipeline.named_steps["classifier"]

print(f"Preprocessor Step: {preprocessor.__class__.__name__}")
for name, trans, cols in preprocessor.transformers_:
    print(f"  - {name} ({trans.__class__.__name__}): {len(cols)} features")

print(f"Classifier Step: {classifier.__class__.__name__}")
print(f"  - Hyperparameters: {classifier.get_params(deep=False)}")
print(f"  - Target Classes: {list(classifier.classes_)}")
print(f"Total Transformed Features at Classifier Input: {len(preprocessor.get_feature_names_out())}")

---
### 4. Reference Evaluation Dataset & Leakage Safeguard Verification

To generate authentic explainability results without synthetic data or model re-fitting, we load the authentic held-out test split ($N=130$) established in Day 16.

**Leakage Safeguards Enforced:**
- Final examination mark `G3` is strictly excluded from feature matrix $X$.
- The derived target `performance_category` is strictly separated from inputs.
- Only the 30 canonical socio-demographic and academic attributes are provided to the pipeline.

In [ ]:
X_test, y_test = load_reference_dataset(data_path)

print("Reference Dataset Inspection:")
print(f"  - Input Feature Matrix (X) Shape: {X_test.shape}")
print(f"  - Target Series (y) Shape: {y_test.shape}")

print("Class Distribution in Held-Out Test Set:")
print(y_test.value_counts())

print("\nLeakage Checks:")
for col in FORBIDDEN_LEAKAGE_COLUMNS:
    print(f"  - Forbidden column '{col}' in X: {col in X_test.columns} [PASSED]")

validate_feature_schema(X_test)
print("Schema Validation Status: PASSED (30/30 features verified)")

---
### 5. Permutation Feature Importance on Held-Out Test Data

#### Methodology
Permutation feature importance (`sklearn.inspection.permutation_importance`) assesses the dependency of the model on each input feature by measuring the degradation in the evaluation metric (**Macro F1**) when the values of that feature are randomly permuted across the test set.

#### Why Permute on Raw Input Columns (Pre-Pipeline)?
Because our model is wrapped in an end-to-end `Pipeline`, we permute raw columns before one-hot encoding. This avoids creating synthetic, invalid categorical states (e.g. permuting one-hot dummy flags independently could result in a student having multiple mother's jobs or none at all).

In [ ]:
perm_df = compute_permutation_importance(
    pipeline=pipeline,
    X=X_test,
    y=y_test,
    scoring="f1_macro",
    n_repeats=15,
    random_state=42,
    n_jobs=1
)

print("=" * 70)
print("TOP 10 FEATURES BY PERMUTATION IMPORTANCE (HELD-OUT TEST SET)")
print("=" * 70)
print(f"{'Rank':<5} {'Feature':<15} {'Mean Macro F1 Drop':<20} {'Std Dev'}")
print("-" * 70)
for _, row in perm_df.head(10).iterrows():
    print(f"{int(row['rank']):<5} {row['feature']:<15} {row['importance_mean']:+.4f}{'':<13} ±{row['importance_std']:.4f}")
print("=" * 70)

---
### 6. Native Feature Importance (Mean Decrease in Impurity)

The fitted `DecisionTreeClassifier` calculates feature importance based on Gini impurity reduction (MDI) across internal tree splits. Here we extract importances across all 56 transformed columns and aggregate them back to the 30 canonical feature names.

In [ ]:
df_transformed, df_native_agg = compute_native_feature_importance(pipeline)

print("=" * 70)
print("DECISION TREE NATIVE FEATURE IMPORTANCE (MDI / GINI IMPURITY)")
print("=" * 70)
print(f"Total Features Analyzed: {len(df_native_agg)} aggregated, {len(df_transformed)} transformed")
print(f"Sum of Native Feature Importances: {df_native_agg['native_importance'].sum():.4f}\n")

print(f"{'Rank':<5} {'Feature':<15} {'Native Importance (Share of Impurity Reduction)'}")
print("-" * 70)
for _, row in df_native_agg.head(10).iterrows():
    print(f"{int(row['rank']):<5} {row['feature']:<15} {row['native_importance']:.4f}")
print("=" * 70)

---
### 7. Visualizing Global Feature Importance

We visualize both **Permutation Feature Importance** and **Native Decision Tree Importance**.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Permutation Importance Plot
top_perm = perm_df.head(10).iloc[::-1]
y_pos = np.arange(len(top_perm))
bars1 = ax1.barh(y_pos, top_perm["importance_mean"], xerr=top_perm["importance_std"], color="#2563EB", alpha=0.85, capsize=4)
ax1.set_yticks(y_pos)
ax1.set_yticklabels(top_perm["feature"], fontsize=10)
ax1.set_xlabel("Mean Macro F1 Decrease (±1 Std Dev)", fontsize=10, fontweight="bold")
ax1.set_title("Permutation Importance (Held-Out Test Set)", fontsize=11, fontweight="bold")
ax1.axvline(0, color="#64748B", linestyle="--")
ax1.grid(axis="x", linestyle="--", alpha=0.35)
for bar in bars1:
    w = bar.get_width()
    ax1.text(w + 0.003 if w >= 0 else w - 0.003, bar.get_y() + bar.get_height()/2, f"{w:+.4f}", va="center", ha="left" if w >= 0 else "right", fontsize=8, fontweight="bold")

# Native MDI Plot
top_nat = df_native_agg.head(10).iloc[::-1]
bars2 = ax2.barh(y_pos, top_nat["native_importance"], color="#0D9488", alpha=0.85)
ax2.set_yticks(y_pos)
ax2.set_yticklabels(top_nat["feature"], fontsize=10)
ax2.set_xlabel("Mean Decrease in Impurity (Gini Importance)", fontsize=10, fontweight="bold")
ax2.set_title("Native Decision Tree MDI (Training Splits)", fontsize=11, fontweight="bold")
ax2.grid(axis="x", linestyle="--", alpha=0.35)
for bar in bars2:
    w = bar.get_width()
    ax2.text(w + 0.002, bar.get_y() + bar.get_height()/2, f"{w:.4f}", va="center", ha="left", fontsize=8, fontweight="bold")

plt.tight_layout()
plt.show()

---
### 8. Explaining Individual Predictions: Decision Paths and Cohort Baselines

To explain individual predictions faithfully without making unsupported causal claims, we trace the exact decision sequence through the fitted tree and compare actionable attributes against reference cohort averages.

In [ ]:
print("=" * 70)
print("LOCAL PREDICTION EXPLANATIONS: REPRESENTATIVE TEST SAMPLES")
print("=" * 70)

for tier in ["Low", "Medium", "High"]:
    indices = y_test[y_test == tier].index
    if len(indices) == 0:
        continue
    idx = indices[0]
    rec = X_test.loc[idx]
    exp = explain_individual_prediction(pipeline, rec, reference_df=X_test)

    print(f"\n[Case Study: {tier} Tier Student (Index {idx})]")
    print(f"Actual Tier: {tier} | Predicted Tier: {exp['predicted_category']}")
    print(f"Leaf Probabilities: {exp['predicted_probabilities']}")
    print(f"Traversed Tree Depth: {exp['traversed_depth']} split nodes")
    print("Evaluated Decision Path:")
    for i, rule in enumerate(exp["decision_rules"][:6], 1):
        print(f"  {i}. Node {rule['node_id']}: {rule['rule_description']}")
    print("Cohort Comparison:")
    for feat in ["failures", "absences", "studytime", "Dalc"]:
        stat = exp["cohort_comparison"][feat]
        print(f"  - {feat}: Student = {stat['student_value']} | Cohort Mean = {stat.get('cohort_mean', '—')}")
    break  # Display one in notebook for brevity

---
### 9. Development Log, Findings, and Methodological Limitations

#### Summary of Findings
1. **`failures` is the Dominant Generalization Driver:** Shuffling `failures` incurs a **$-0.0629$ drop in Macro F1** on unseen test data. Past academic difficulty remains the strongest early indicator.
2. **Socio-Demographic Disparities Matter:** Mother's occupation (`Mjob`, $+0.0437$) and family relations (`famrel`, $+0.0166$) demonstrate substantial predictive contribution.
3. **Generalization Discrepancies in Tree Splitting:** Attributes like `health` and `reason` have high Gini impurity reduction on training data but near-zero permutation importance on held-out test data. This illustrates the risk of relying solely on tree MDI.

#### Critical Limitations & Ethical Constraints
- **Association $\neq$ Causation:** High feature importance indicates that a variable has high predictive power within this model. It does not prove that changing the variable will cause grades to improve.
- **Collinear Surrogacy:** Correlated features (e.g. `Medu` and `Fedu`, or `Dalc` and `Walc`) dilute individual permutation importance scores.
- **Sample Size Constraints:** Held-out test set size ($N=130$) carries inherent statistical variance.